# Comprobación del modelo de reconocimiento de robots

In [3]:
import yaml
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import supervision as sv
from ultralytics import YOLO, SAM

print("Todo importado correctamente")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No hay GPU")

Todo importado correctamente
GPU: NVIDIA GeForce RTX 5050


In [4]:
model = YOLO("yucabot.pt")
print("Clases:", model.names)

Clases: {0: 'ball', 1: 'robot'}


## Practica con el modelo

In [5]:
import warnings
warnings.filterwarnings("ignore")

from ultralytics import YOLO, SAM
import supervision as sv
import numpy as np

# ── Modelos ───────────────────────────────────────────────────────────────────
yolo_model = YOLO("yucabot.pt")
sam_model  = SAM("sam3.pt")

# ── Anotadores ────────────────────────────────────────────────────────────────
mask_annotator  = sv.MaskAnnotator(opacity=0.5)
box_annotator   = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()
trace_annotator = sv.TraceAnnotator()

# ── Tracker y memoria ─────────────────────────────────────────────────────────
tracker = sv.ByteTrack()
tracker.reset()

ultima_deteccion_valida = None
frames_sin_deteccion = 0
MAX_FRAMES_SIN_DETECCION = 15

def procesar_frame_yucabot(frame: np.ndarray, _: int) -> np.ndarray:
    global ultima_deteccion_valida, frames_sin_deteccion

    yolo_results = yolo_model(frame, verbose=False)[0]
    yolo_det = sv.Detections.from_ultralytics(yolo_results)
    yolo_det = yolo_det[yolo_det.confidence >= 0.50]

    if len(yolo_det) > 0:
        bboxes = yolo_det.xyxy.tolist()
        try:
            sam_results = sam_model(frame, bboxes=bboxes, imgsz=1036, verbose=False)[0]
            sam_det = sv.Detections.from_ultralytics(sam_results)
        except Exception:
            sam_det = sv.Detections.empty()

        yolo_det = tracker.update_with_detections(yolo_det)

        usar_sam = len(sam_det) == len(yolo_det) and len(sam_det) > 0
        det_final = sam_det if usar_sam else yolo_det

        det_final.tracker_id = yolo_det.tracker_id
        det_final.class_id   = yolo_det.class_id
        det_final.confidence = yolo_det.confidence

        ultima_deteccion_valida = det_final
        frames_sin_deteccion = 0

    else:
        frames_sin_deteccion += 1
        if ultima_deteccion_valida is not None and frames_sin_deteccion <= MAX_FRAMES_SIN_DETECCION:
            det_final = ultima_deteccion_valida
        else:
            return frame

    labels = [
        f"{yolo_model.names[cid]} ID:{tid} ({conf:.0%})"
        for tid, cid, conf in zip(
            det_final.tracker_id,
            det_final.class_id,
            det_final.confidence
        )
    ] if det_final.tracker_id is not None else []

    usar_mascaras = hasattr(det_final, 'mask') and det_final.mask is not None
    if usar_mascaras:
        annotated = mask_annotator.annotate(scene=frame.copy(), detections=det_final)
    else:
        annotated = box_annotator.annotate(scene=frame.copy(), detections=det_final)

    if labels:
        annotated = label_annotator.annotate(scene=annotated, detections=det_final, labels=labels)

    annotated = trace_annotator.annotate(scene=annotated, detections=det_final)

    return annotated


# ── Procesar video ────────────────────────────────────────────────────────────
sv.process_video(
    source_path="../VideosEjemplo/gael.mov",
    target_path="assets/yucabot_v1.mp4",
    callback=procesar_frame_yucabot
)
print("✓ Guardado: assets/yucabot_v1.mp4")

✓ Guardado: assets/yucabot_v1.mp4
